# NeuralZip: transparent Hugging Face compression

Activate NeuralZip once, then use the ordinary Transformers API. An ordinary BF16 checkpoint is compressed automatically with the active precomputed setup; a saved NeuralZip checkpoint restores the complete lossless storage representation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from neuralzip import neuralzip

# Activate once per Python process. This uses the bundled global codebook.
# Pass a trusted precomputed setup to reuse family-specific codebooks.
neuralzip()

## Load and run an ordinary checkpoint

Hugging Face loads the model normally. NeuralZip then compresses eligible BF16 weights with the active setup and attaches them for sequential inference on the selected GPU. With `setup=None`, the bundled global BF16 codebook is used without analyzing the model to create a new setup.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
stats = model.neuralzip_stats
print(f"Encode: {stats.encoding_seconds:.3f} s")
print(f"Ratio: {stats.compression_ratio:.3f}")
print(f"Exponent ratio: {stats.exponent_ratio:.3f}")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device="cuda:0",
)
print(generator("Machine learning is", max_new_tokens=20, do_sample=False))

## Save and reload the compressed checkpoint

The standard Hugging Face save method runs NeuralZip's full storage codec and writes its raw, shared-Huffman, or packed streams to Safetensors. The model's statistics then describe that persistent representation. A later standard load restores every tensor bit-for-bit and prepares eligible BF16 weights for GPU inference; it does not run setup again. Call `neuralzip()` once at the start of every new process before loading.

In [ ]:
model.save_pretrained("qwen-neuralzip")
tokenizer.save_pretrained("qwen-neuralzip")
print(model.neuralzip_stats)

del generator, model

loaded_model = AutoModelForCausalLM.from_pretrained("qwen-neuralzip")
print(f"Decode: {loaded_model.neuralzip_stats.decoding_seconds:.3f} s")

## Optional: prepare a reusable family setup

The executable example above uses the global fast path. Creating a custom setup is a separate one-time workflow run before activating NeuralZip:

```python
import torch

reference = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)
setup = neuralzip.setup(reference)
setup.save("qwen-family.setup.pkl")
```

This is the complete setup pass: it analyzes all floating FP16, BF16, and FP32 state and considers raw, shared-Huffman, and packed exponent modes. In a later process, call `neuralzip("qwen-family.setup.pkl")` and load compatible models with the ordinary Transformers API. The wrapper performs the encode automatically; you can also activate the in-memory object directly with `neuralzip(setup)`.